In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer # type: ignore
from tensorflow.keras.preprocessing.sequence import pad_sequences # type: ignore
from tensorflow.keras.models import Model # type: ignore
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense # type: ignore

In [2]:
sentences = [
 "I love this product",
 "This movie made me smile",
 "Service was friendly and quick",
 "Today felt bright and happy",
 "This is the best day",
 "Absolutely fantastic experience",
 "I enjoyed every single moment",
 "Great job, well done",
 "The food tasted delicious",
 "Totally recommend to everyone",
 "Very satisfied with results",
 "This worked better than expected",
 "Amazing quality and value",
 "Such a pleasant surprise",
 "I feel positive about this",
 "I hate this product",
 "This movie bored me",
 "Service was rude and slow",
 "Today was cold and lonely",
 "This is the worst day",
 "Terrible experience overall",
 "I regret buying this",
 "Very disappointed with results",
 "The food tasted awful",
 "Do not recommend this",
 "It broke after one use",
 "Not worth the money",
 "Utterly frustrating and annoying",
 "I feel negative about this",
 "Such a waste of time",
]
labels = [1]*15 + [0]*15
labels = np.array(labels)

In [3]:
vocab_size = 2000 #Most frequent words in Dataset , Reduces Complexity , embedding layer size , prevents rare words 
tok  = Tokenizer(num_words=vocab_size,oov_token="<OOV>") #OOV(Out of Vocabulary) , Num_words = keep only top 2000 most frequent words
tok.fit_on_texts(sentences) #Build words Index I[1],am[2],a[3],man[4]
seqs = tok.texts_to_sequences(sentences)#Converts text integer sequences
maxlen = max(len(s) for s in seqs)#Finds length of longest sequence 
X = pad_sequences(seqs,maxlen = maxlen,padding='post')#padding = Add Zeros at the end of shorter sequences 
y = labels #output label

In [4]:
maxlen

5

In [5]:
X[0]

array([ 3, 26,  2,  7,  0], dtype=int32)

In [6]:
embed_dim = 16
rnn_units = 8

In [7]:
inp = Input(shape=(maxlen,),dtype="int32",name='input')
x = Embedding(input_dim = vocab_size,output_dim = embed_dim,mask_zero = True,name = 'embed')(inp)
rnn = SimpleRNN(units = rnn_units,return_sequences = False,return_state = False,name = 'simple_rnn')
x_last = rnn(x)
out = Dense(1,activation='sigmoid',name='out')(x_last)
model = Model(inputs = inp,outputs = out )
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 5)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embed (Embedding)   │ (None, 5, 16)     │     32,000 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 5)         │          0 │ input[0][0]       │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn          │ (None, 8)         │        200 │ embed[0][0],      │
│ (SimpleRNN)         │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ out (Dense)         │ (None, 1)         │          9 │ simple_rnn[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 32,209 (125.82 KB)

 Trainable params: 32,209 (125.82 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
model.fit(X,y,epochs = 25,batch_size = 8,verbose = 1)

Epoch 1/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.6000 - loss: 0.6828
Epoch 2/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.7000 - loss: 0.6631 
Epoch 3/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.8667 - loss: 0.6446 
Epoch 4/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.9000 - loss: 0.6272
Epoch 5/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.9333 - loss: 0.6058 
Epoch 6/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 1.0000 - loss: 0.5861 
Epoch 7/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 1.0000 - loss: 0.5645 
Epoch 8/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 1.0000 - loss: 0.5406 
Epoch 9/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 1.0000 - loss: 0.5166 
Epoch 10/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 1.0000 - loss: 0.4900 
Epoch 11/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 1.0000 - loss: 0.4631 
Epoch 12/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 1.0000 - loss:

In [10]:
intermediate_model = Model(inputs = model.input, outputs = [model.get_layer('embed').output, model.get_layer('simple_rnn').output] )  

In [12]:
from tensorflow.keras.layers import SimpleRNN as SRNN
seq_inp = Input(shape = (maxlen,),dtype='int32')
seq_emb = model.get_layer('embed')(seq_inp)

rnn_seq = SRNN(units = rnn_units, return_sequences = True, name = 'rnn_seq') # Create RNN with return_sequences=True

seq_hidden = rnn_seq(seq_emb)# builds automatically

try:
    trained_weights = model.get_layer('simple_rnn').get_weights()
    rnn_seq.set_weights(trained_weights)
    print("Copied RNN weights into sequence-inspection RNN.")
except Exception as e:
    print("Could not copy weights automatically:", e)

inspect_model = Model(inputs=seq_inp, outputs=seq_hidden)

idx = 0
example_seq = X[idx:idx+1]  # shape (1, maxlen)
hidden_seq = inspect_model.predict(example_seq)

print("Sentence:", sentences[idx])
print("Token ids:", example_seq)
print("Hidden states per timestep shape:", hidden_seq.shape)
print("Hidden states (timesteps x units):")
print(np.round(hidden_seq[0], 3))

Copied RNN weights into sequence-inspection RNN.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
Sentence: I love this product
Token ids: [[ 3 26  2  7  0]]
Hidden states per timestep shape: (1, 5, 8)
Hidden states (timesteps x units):
[[-0.055  0.026  0.066 -0.02  -0.035  0.016 -0.031  0.096]
 [ 0.129 -0.13  -0.099  0.058  0.121 -0.169 -0.212  0.016]
 [ 0.107 -0.024  0.253 -0.062 -0.316  0.143 -0.018  0.464]
 [-0.125  0.492 -0.301  0.281  0.149  0.068  0.294 -0.313]
 [-0.125  0.492 -0.301  0.281  0.149  0.068  0.294 -0.313]]
